In [7]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [8]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
LATRANGE  = CONFIGS['domain']['latrange']
LONRANGE  = CONFIGS['domain']['lonrange']
SPLIT     = 'valid'

In [9]:
# Load z-score stats and normalized split
with open(os.path.join(SPLITSDIR, 'stats.json'), 'r', encoding='utf-8') as f:
    stats = json.load(f)

with xr.open_dataset(os.path.join(SPLITSDIR, f'norm_{SPLIT}.h5'), engine='h5netcdf') as ds:
    lf_z  = ds['lf'].load()   # (lat, lon) — static land fraction, z-scored
    shf_z = ds['shf'].load()  # (lat, lon, time) — sensible heat flux, z-scored

# Broadcast static LF to match SHF time dimension
lf_z_full = lf_z.expand_dims(time=shf_z.time).broadcast_like(shf_z)

# Derived quantities for SR-HI: max(lf_z, shf_z) · d
max_z      = xr.apply_ufunc(np.maximum, lf_z_full, shf_z)
lf_wins    = (lf_z_full > shf_z).mean('time')  # fraction where LF dominates

# Time means
shf_z_mean = shf_z.mean('time')
max_z_mean = max_z.mean('time')

# Native units for reference panels
lf_mean_val  = stats.get('lf_mean', 0.0);  lf_std_val  = stats.get('lf_std', 1.0)
shf_mean_val = stats.get('shf_mean', 0.0); shf_std_val = stats.get('shf_std', 1.0)
lf_native  = lf_z  * lf_std_val  + lf_mean_val
shf_native = shf_z_mean * shf_std_val + shf_mean_val

lf_wins_pct = 100 * float((lf_z_full.values > shf_z.values).mean())
print(f'LF wins at {lf_wins_pct:.1f}% of all (time, lat, lon) samples')

LF wins at 30.2% of all (time, lat, lon) samples


In [ ]:
import pickle, os
# Load optimized d coefficient for SR-HI if available (for the bottom-right panel)
_pkl = os.path.join(CONFIGS['filepaths']['models'], 'sr', 'optimized_equations.pkl')
if os.path.exists(_pkl):
    with open(_pkl, 'rb') as _f:
        _reg = pickle.load(_f)
    _d = _reg.get('sr_hi', {}).get('constants', {}).get('d', None)
else:
    _d = None

# ── 2×2 figure ──────────────────────────────────────────────────────────────
fig, axs = pplt.subplots(nrows=2, ncols=2, proj='cyl', figwidth=6.5,
                          share=False, wspace=0.3, hspace=0.4)
axs.format(coast=True, borders=False, latlim=LATRANGE, lonlim=LONRANGE,
           latlines=5, lonlines=5, grid=False)

# Panel (a): LF in native units — static land/sea geography
m0 = axs[0].pcolormesh(lf_native.lon, lf_native.lat, lf_native,
                        cmap='Browns', vmin=0, vmax=1, levels=11)
axs[0].format(title='(a) Land fraction (LF)')
fig.colorbar(m0, ax=axs[0], loc='b', label='fraction', ticks=0.2)

# Panel (b): SHF JJA mean in native units — ocean cooling vs land heating
shf_lim = max(abs(float(shf_native.min())), abs(float(shf_native.max())))
m1 = axs[1].pcolormesh(shf_native.lon, shf_native.lat, shf_native,
                        cmap='ColdHot', vmin=-shf_lim, vmax=shf_lim, levels=21)
axs[1].format(title='(b) Sensible heat flux — JJA mean')
fig.colorbar(m1, ax=axs[1], loc='b', label='W m$^{-2}$')

# Panel (c): max(LF_z, SHF_z) time-mean — what SR-HI actually receives
vmax_max = max(abs(float(max_z_mean.min())), float(max_z_mean.quantile(0.99)))
m2 = axs[2].pcolormesh(max_z_mean.lon, max_z_mean.lat, max_z_mean,
                        cmap='ColdHot', vmin=-vmax_max, vmax=vmax_max, levels=21)
axs[2].format(title='(c) max$(\\widetilde{LF},\\,\\widetilde{SHF})$ — JJA mean')
fig.colorbar(m2, ax=axs[2], loc='b', label='std. units')

# Panel (d): LF-wins fraction — where LF vs SHF dominates the max
m3 = axs[3].pcolormesh(lf_wins.lon, lf_wins.lat, lf_wins,
                        cmap='DryWet', vmin=0, vmax=1, levels=11)
axs[3].format(title='(d) Fraction of time: LF > SHF')
fig.colorbar(m3, ax=axs[3], loc='b', label='fraction', ticks=0.2)

pplt.show()
# fig.save('../figs/fig_surface_terms.jpg')

## Interpretation

**Panel (a) — Land fraction (LF):** A static geography field ranging from 0 (open ocean) to 1 (inland). It is highest over the Indian subcontinent, the Arabian Peninsula, and the Tibetan Plateau — precisely the land masses that bound the South Asian monsoon domain.

**Panel (b) — Sensible heat flux (SHF), JJA mean:** SHF is *negative* over the Indian Ocean and Bay of Bengal (the ocean surface cools the overlying atmosphere during boreal summer) and *positive* over hot land surfaces. The large negative excursions over the ocean are what drives the asymmetric z-score ranges seen below.

**Panel (c) — max(LF̃, SHF̃), JJA mean:** This is the surface-flux term that SR-HI ingests. Because SHF has wide variability (σ ≈ 40 W m⁻²) and is persistently *negative* over the ocean, the normalized land-fraction LF̃ — whose range is narrow (−0.66 to +1.60 std units) — frequently *wins* the max, even over oceanic grid points where LF ≈ 0. The result is a field that is positively biased over land and near-zero to slightly positive over ocean.

**Panel (d) — Fraction of time LF̃ > SHF̃:** LF̃ dominates at ~30 % of all (time, lat, lon) samples globally, but the spatial pattern is revealing: LF̃ wins consistently over the open ocean (where SHF_z is deeply negative) and is contested over the subcontinent (where SHF_z can be large and positive). This means the surface term primarily acts as a *land-mask proxy* in the current optimized solution.

**Physical interpretation of the SR-HI surface modifier:** The optimized coefficient d (and thus γ_SFC = d·σ_θe) can be negative, meaning that larger max(LF̃, SHF̃) *suppresses* precipitation onset in normalized θe units. This is consistent with the monsoon: heavy rainfall is concentrated over the warm ocean and coastal zones, while the hot, dry interior land surfaces — despite high LF and positive SHF — are less favorable for deep convection. The SR-HI equation therefore learns to partially offset the atmospheric instability criterion (θe branch) with a surface-flux penalty over land-dominant grid points.

In [12]:
# Crossover diagnostics: when does LF_z exceed SHF_z and by how much?
lf_arr  = lf_z_full.values.ravel()
shf_arr = shf_z.values.ravel()
fin     = np.isfinite(lf_arr) & np.isfinite(shf_arr)

print(f'LF wins at {lf_wins_pct:.1f}% of all (time, lat, lon) samples')
print(f'lf_z  : mean={lf_arr[fin].mean():.3f}  std={lf_arr[fin].std():.3f}  '\
      f'range=[{lf_arr[fin].min():.2f}, {lf_arr[fin].max():.2f}]')
print(f'shf_z : mean={shf_arr[fin].mean():.3f}  std={shf_arr[fin].std():.3f}  '\
      f'range=[{shf_arr[fin].min():.2f}, {shf_arr[fin].max():.2f}]')

# Native LF value at the crossover point (mean SHF_z level)
shf_z_mean_val = float(shf_arr[fin].mean())
lf_crossover   = shf_z_mean_val * lf_std_val + lf_mean_val
print(f'\nCrossover: shf_z global mean = {shf_z_mean_val:.3f} → native LF ≈ {lf_crossover:.3f}')

LF wins at 30.2% of all (time, lat, lon) samples
lf_z  : mean=-0.000  std=1.000  range=[-0.66, 1.60]
shf_z : mean=0.014  std=0.958  range=[-11.67, 3.24]

Crossover: shf_z global mean = 0.014 → native LF ≈ 0.299
